# 🛡️ Hate Speech Detection
Multilingual BERT + Whisper ASR

## 1. Install Dependencies

In [1]:
!pip install -q gTTS openai-whisper datasets transformers torch torchaudio pydub nltk scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 46.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.6 MB/s eta 0:00:00


In [2]:
!pip install schedule

## 2. Imports

In [8]:
import re
import numpy as np
import pandas as pd
import nltk
import torch
import whisper
import schedule
import time

from datasets import load_dataset
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

import torch.nn.functional as F
from gtts import gTTS
from tqdm import tqdm

nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## 3. Load Datasets

In [9]:
dataset1 = load_dataset("hate_speech_offensive")

df1 = dataset1["train"].to_pandas()[["tweet","class"]]
df1.columns = ["text","label"]

df1["label"] = df1["label"].map({
0:"hate",
1:"offensive",
2:"safe"
})

In [10]:
dataset2 = load_dataset("tweet_eval","hate")

df2 = dataset2["train"].to_pandas()[["text","label"]]

df2["label"] = df2["label"].map({
1:"hate",
0:"safe"
})

In [11]:
dataset3 = load_dataset("tweet_eval","offensive")

df3 = dataset3["train"].to_pandas()[["text","label"]]

df3["label"] = df3["label"].map({
1:"offensive",
0:"safe"
})

In [12]:
df_all = pd.concat([df1,df2,df3],ignore_index=True)

print(df_all["label"].value_counts())

label
offensive    23131
safe         17355
hate          5213
Name: count, dtype: int64


## 4. Preprocessing
- Lowercase
- Remove URLs
- Remove punctuation
- Remove extra spaces
- Remove short words
- Remove stopwords (EN)

In [13]:
stop_words_en = set(stopwords.words("english"))

def preprocess(text):

    text = str(text).lower()

    text = re.sub(r"http\S+|www\S+","",text)

    text = re.sub(r"@\w+","",text)

    text = re.sub(r"#","",text)

    text = re.sub(r"\d+","",text)

    text = re.sub(r"[^\w\s]","",text)

    text = re.sub(r"\s+"," ",text).strip()

    return text

df_all["text"] = df_all["text"].apply(preprocess)

df_all = df_all[df_all["text"]!=""]

In [14]:
label_map = {
"hate":0,
"offensive":1,
"safe":2
}

df_all["label"] = df_all["label"].map(label_map)

## 6. Train / Test Split

In [15]:
train_texts, test_texts, train_labels, test_labels = train_test_split(

df_all["text"].tolist(),
df_all["label"].tolist(),

test_size=0.2,
random_state=42,
stratify=df_all["label"]
)

## 7. Tokenization

In [16]:
MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts):

    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=160,
        return_tensors="pt"
    )

train_enc = tokenize(train_texts)
test_enc  = tokenize(test_texts)

train_dataset = TensorDataset(
train_enc["input_ids"],
train_enc["attention_mask"],
torch.tensor(train_labels)
)

test_dataset = TensorDataset(
test_enc["input_ids"],
test_enc["attention_mask"],
torch.tensor(test_labels)
)

train_loader = DataLoader(train_dataset,batch_size=16,shuffle=True)
test_loader  = DataLoader(test_dataset,batch_size=16)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 8. Load Model + Weighted Loss

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
MODEL_NAME,
num_labels=3
)

model.to(device)

optimizer = optim.AdamW(model.parameters(),lr=2e-5)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
class_weights = compute_class_weight(

class_weight="balanced",

classes=np.array([0,1,2]),

y=df_all["label"]

)

weights = torch.tensor(class_weights,dtype=torch.float).to(device)

loss_fn = nn.CrossEntropyLoss(weight=weights)

In [19]:
EPOCHS = 7

total_steps = len(train_loader)*EPOCHS

scheduler = get_linear_schedule_with_warmup(

optimizer,

num_warmup_steps=0,

num_training_steps=total_steps
)

## 9. Training

In [20]:
best_acc = 0
patience = 2
epochs_no_improve = 0

for epoch in range(EPOCHS):

    model.train()

    correct = 0
    total = 0
    total_loss = 0

    for input_ids,attention_mask,labels in tqdm(train_loader):

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(input_ids,attention_mask=attention_mask)

        loss = loss_fn(outputs.logits,labels)

        loss.backward()

        optimizer.step()

        scheduler.step()

        total_loss += loss.item()

        preds = outputs.logits.argmax(dim=1)

        correct += (preds==labels).sum().item()

        total += labels.size(0)

    acc = correct/total*100

    print(f"Epoch {epoch+1} Loss:{total_loss/len(train_loader):.4f} Acc:{acc:.2f}%")

    if acc > best_acc:

        best_acc = acc
        epochs_no_improve = 0

    else:

        epochs_no_improve += 1

    if epochs_no_improve == patience:

        print("Early stopping")

        break

100%|██████████| 2284/2284 [11:30<00:00,  3.31it/s]


Epoch 1 Loss:0.6753 Acc:74.35%


100%|██████████| 2284/2284 [11:34<00:00,  3.29it/s]


Epoch 2 Loss:0.5003 Acc:80.18%


100%|██████████| 2284/2284 [11:34<00:00,  3.29it/s]


...

Epoch 5 Loss:0.3013 Acc:87.61%


100%|██████████| 2284/2284 [11:34<00:00,  3.29it/s]


Epoch 6 Loss:0.2464 Acc:89.94%


100%|██████████| 2284/2284 [11:34<00:00,  3.29it/s]

Epoch 7 Loss:0.2064 Acc:91.69%


## 10. Evaluation

In [21]:
model.eval()

all_preds=[]
all_labels=[]

with torch.no_grad():

    for input_ids,attention_mask,labels in test_loader:

        input_ids=input_ids.to(device)
        attention_mask=attention_mask.to(device)

        outputs=model(input_ids,attention_mask=attention_mask)

        preds=outputs.logits.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(

all_labels,
all_preds,

target_names=["hate","offensive","safe"]

))

              precision    recall  f1-score   support

        hate       0.53      0.71      0.61      1042
   offensive       0.88      0.82      0.85      4626
        safe       0.85      0.84      0.84      3468

    accuracy                           0.82      9136
   macro avg       0.75      0.79      0.77      9136
weighted avg       0.83      0.82      0.82      9136



In [22]:
def classify_text_confidence(text):

    enc = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=160
    )

    enc = {k:v.to(device) for k,v in enc.items()}

    with torch.no_grad():

        logits = model(**enc).logits

        probs = F.softmax(logits,dim=1).cpu().numpy()[0]

        pred = probs.argmax()

    label_names = {

    0:"hate",
    1:"offensive",
    2:"safe"

    }

    return label_names[pred],probs

In [23]:
examples = [

"I hate you",

"You are stupid",

"I love this course"

]

for e in examples:

    label,probs = classify_text_confidence(e)

    print(e)
    print(label)
    print(probs)
    print("-----")

I hate you
hate
[0.9280221  0.02744696 0.04453089]
-----
You are stupid
offensive
[4.6386578e-04 9.9103290e-01 8.5031893e-03]
-----
I love this course
safe
[1.5359861e-04 6.8685721e-04 9.9915946e-01]
-----


## 11. Save Model

In [24]:
model.save_pretrained("./hate_speech_model")
tokenizer.save_pretrained("./hate_speech_model")
print("Model saved to ./hate_speech_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./hate_speech_model


## 12. Audio Pipeline (ASR → Classify)

In [31]:
whisper_model = whisper.load_model("small")

def audio_to_clean_text(audio_path):
    """Transcribe audio with Whisper and clean the output."""
    try:
        result = whisper_model.transcribe(audio_path)
        transcription = result["text"]
        print(f"Transcription: {transcription}")
        return preprocess(transcription)
    except Exception as e:
        print(f"Error during ASR: {e}")
        return ""

def classify_text(text):
    """Run hate speech classification on a cleaned text string."""
    if not text.strip():
        print("Empty text — cannot classify.")
        return None

    enc = tokenizer(text, truncation=True, padding=True, max_length=128, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}

    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits
        pred   = logits.argmax(dim=1).item()

    label_names = {0: "hate", 1: "offensive", 2: "safe"}
    return label_names[pred]

def classify_audio(audio_path):
    """End-to-end: audio file → hate speech label."""
    cleaned_text = audio_to_clean_text(audio_path)
    print(f"Cleaned text: {cleaned_text}")
    label = classify_text(cleaned_text)
    print(f"Prediction: {label}")
    return label

## 13. Test with gTTS
> **Note:** gTTS saves as MP3. Whisper reads MP3 natively — no conversion needed.

In [32]:
test_sentence = "Hey! Could you check out this course at https://example.com? I don't understand. "
tts = gTTS(text=test_sentence, lang="en")
tts.save("test_audio.mp3")

result = classify_audio("test_audio.mp3")
print(f"\nFinal label for test audio: {result}")

Transcription:  Hey! Could you check out this course at https-slash-example.com? I don't understand.
Cleaned text: hey could you check out this course at i dont understand
Prediction: safe

Final label for test audio: safe
